In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"

import random, glob
import numpy as np
import torch
import torchaudio
from transformers import Qwen2AudioForConditionalGeneration, AutoProcessor
from peft import PeftModel
from sklearn.metrics import classification_report, f1_score
from collections import Counter
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

BASE_DIR = os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath("__file__"))))
TARGET_SR = 16000
MAX_SAMPLES = 30 * TARGET_SR

print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

/data/liharrison/miniconda3/envs/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


In [2]:
# Load base model + LoRA adapters
LORA_DIR = os.path.join(BASE_DIR, "data", "models", "qwen2_audio_lora")

processor = AutoProcessor.from_pretrained(LORA_DIR)
base_model = Qwen2AudioForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2-Audio-7B-Instruct",
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model = PeftModel.from_pretrained(base_model, LORA_DIR)
model.eval()
print(f"Loaded LoRA adapters from {LORA_DIR}")

Loading weights: 100%|██████████| 876/876 [00:04<00:00, 201.90it/s, Materializing param=multi_modal_projector.linear.weight]                           


Loaded LoRA adapters from /data/liharrison/lvsim/data/models/qwen2_audio_lora


In [3]:
SYSTEM_PROMPT = "You are an expert speech-language pathologist."

CONTROL_DESC = (
    "CONTROL — Speech is fluent with no pathological features. "
    "Normal hesitations, filler words, self-corrections, and brief pauses "
    "may be present but are typical of healthy speech.\n"
    "    Example: aɪ səpˈoʊz ðə tɹˈɪp ðæt stˈeɪz wɪð mˌiː mˈoʊst wʌz, "
    "wˈɛl, ɐ kwˈaɪət sˈʌmɚ wiː spˈɛnt ˌʌp æt lˈeɪk plˈæsɪd"
)
LVPPA_DESC = (
    "lvPPA — Speech shows features of logopenic variant primary progressive aphasia, "
    "including abnormally prolonged pauses during word retrieval, "
    "phonological errors (sound-level distortions, substitutions, or false starts), "
    "repetitive attempts at words, and fragmented sentence production.\n"
    "    Example: aɪ səpˈoʊ[PROLONG]z ðə, ðə, ðə θˈɪŋ wiː dˈɪd, "
    "ðə tɹˈɪp ðæt stˈeɪz wɪð mˌiː mˈoʊst, ɪt wʌzɐ kwˈaɪ...kwˈaɪət"
)


def get_prompt(counterbalance: bool = None):
    if counterbalance is None:
        counterbalance = random.random() < 0.5

    if counterbalance:
        a_desc, b_desc = CONTROL_DESC, LVPPA_DESC
        a_label, b_label = "CONTROL", "lvPPA"
    else:
        a_desc, b_desc = LVPPA_DESC, CONTROL_DESC
        a_label, b_label = "lvPPA", "CONTROL"

    prompt = (
        "Listen carefully to this audio of a person speaking.\n\n"
        "Which best describes this speaker?\n"
        f"A. {a_desc}\n\n"
        f"B. {b_desc}\n\n"
        "Respond with only the letter: A or B."
    )

    option_map = {"A": a_label, "B": b_label}
    return prompt, option_map


def classify_audio(audio_path):
    wav, sr = torchaudio.load(audio_path)
    wav = torchaudio.functional.resample(wav, sr, TARGET_SR).mean(0)
    if wav.shape[0] > MAX_SAMPLES:
        wav = wav[:MAX_SAMPLES]
    audio_np = wav.numpy()

    prompt, option_map = get_prompt()

    conversation = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": [
            {"type": "audio", "audio_url": audio_path},
            {"type": "text", "text": prompt},
        ]},
    ]
    text = processor.apply_chat_template(conversation, add_generation_prompt=True, tokenize=False)
    inputs = processor(text=text, audio=[audio_np], sampling_rate=TARGET_SR, return_tensors="pt", padding=True)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        ids = model.generate(**inputs, max_new_tokens=10)
    ids = ids[:, inputs["input_ids"].size(1):]
    response = processor.batch_decode(ids, skip_special_tokens=True)[0].strip()
    return response, option_map


def parse_response(response, option_map):
    r = response.strip().upper()
    if r.startswith("A"):
        return 1 if option_map["A"] == "lvPPA" else 0
    if r.startswith("B"):
        return 1 if option_map["B"] == "lvPPA" else 0
    r_lower = response.lower()
    if "lvppa" in r_lower or "dysfluent" in r_lower:
        return 1
    if any(w in r_lower for w in ["control", "healthy", "normal", "fluent"]):
        return 0
    return -1

In [4]:
REAL_DIR = os.path.join(BASE_DIR, "data", "real")
groups = {
    "lvPPA":     (sorted(glob.glob(os.path.join(REAL_DIR, "*lvPPA*", "**", "*.wav"), recursive=True)), 1),
    "JHU":       (sorted(glob.glob(os.path.join(REAL_DIR, "*jhu*", "**", "*.wav"), recursive=True)), 1),
    "Control":   (sorted(glob.glob(os.path.join(REAL_DIR, "*segmentedcc*", "**", "*.wav"), recursive=True)), 0),
    "Capilouto": (sorted(glob.glob(os.path.join(REAL_DIR, "*Capilouto*", "**", "*.wav"), recursive=True)), 0),
}

for name, (files, _) in groups.items():
    print(f"{name}: {len(files)} clips")

lvPPA: 89 clips
JHU: 74 clips
Control: 235 clips
Capilouto: 311 clips


In [5]:
all_true, all_pred, all_raw = [], [], []

for name, (files, true_label) in groups.items():
    preds, raw_responses = [], []
    for f in tqdm(files, desc=name):
        resp, option_map = classify_audio(f)
        raw_responses.append(resp)
        pred = parse_response(resp, option_map)
        if pred == -1:
            pred = 0
        preds.append(pred)

    correct = sum(1 for p in preds if p == true_label)
    total = len(preds)
    acc = correct / total if total > 0 else 0
    dys_count = sum(1 for p in preds if p == 1)

    all_true.extend([true_label] * total)
    all_pred.extend(preds)
    all_raw.extend(raw_responses)

    print(
        f"{name:>10s}  ({total:3d} clips)  acc={acc:.3f}  "
        f"dys={dys_count}  healthy={total - dys_count}"
    )

lvPPA: 100%|██████████| 89/89 [00:23<00:00,  3.73it/s]


     lvPPA  ( 89 clips)  acc=0.461  dys=41  healthy=48


JHU: 100%|██████████| 74/74 [00:19<00:00,  3.88it/s]


       JHU  ( 74 clips)  acc=0.419  dys=31  healthy=43


Control: 100%|██████████| 235/235 [01:01<00:00,  3.82it/s]


   Control  (235 clips)  acc=0.613  dys=91  healthy=144


Capilouto: 100%|██████████| 311/311 [01:22<00:00,  3.78it/s]

 Capilouto  (311 clips)  acc=0.576  dys=132  healthy=179


In [ ]:
all_true = np.array(all_true)
all_pred = np.array(all_pred)

print("=" * 60)
print("Overall Classification Report (LoRA Fine-Tuned Qwen2-Audio)")
print("=" * 60)
print(classification_report(all_true, all_pred, target_names=["healthy", "dysfluent"]))
print(f"F1 Macro: {f1_score(all_true, all_pred, average='macro'):.4f}")
print(f"Accuracy: {(all_true == all_pred).mean():.4f}")

print("\nRaw response distribution:")
for resp, count in Counter(all_raw).most_common(15):
    print(f"  '{resp}': {count}")